# 01 — ESOM basics

This notebook walks through the core `ESOM` wrapper step by step:

1. Load the **Hepta** FCPS benchmark dataset (212 points · 3 features · 7 clusters)
2. Standardise the data
3. Train a medium-sized ESOM grid (32 × 36 = 1152 neurons)
4. Inspect `weights`, `u_matrix`, `hit_map`, `bmu_indices`
5. Visualise with **Altair** interactive charts
6. Explore component planes

All training is reproducible via `random_seed`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

def load_fcps_fixture(name: str):
    """Load one dataset from tests/fixtures/fcps.npz (walk up from cwd to find repo root)."""
    for d in [Path.cwd(), *Path.cwd().parents]:
        p = d / "tests" / "fixtures" / "fcps.npz"
        if p.is_file():
            z = np.load(p)
            return z[f"{name}_data"].copy(), z[f"{name}_cls"].copy()
    raise FileNotFoundError(
        "tests/fixtures/fcps.npz not found — run Jupyter from inside the pyesom repo checkout."
    )

from pyesom import ESOM
from pyesom.visualization.component import plot_component_planes
from pyesom.visualization.heatmap import plot_grid_heatmap

## 1  Load & standardise Hepta

Data load from **`tests/fixtures/fcps.npz`** (FCPS export, GPL-3). U\*F on Atom/Lsun/WingNut/… is benchmarked in Moutarde & Ultsch (2005); **Hepta** is used elsewhere as the FCPS seven-blobs toy (e.g. Thrun 2018, Fig. 5.1). Expect methodological—not bitwise—agreement with those implementations (different ESOM trainer, grid sizes).

We z-score each feature so all dimensions contribute equally to distances.

In [2]:
data, true_labels = load_fcps_fixture("hepta")
print(f"data shape  : {data.shape}   (n_samples, n_features)")
print(f"true labels : {np.unique(true_labels).tolist()}")
print(f"n_clusters  : {len(np.unique(true_labels))}")

data shape  : (212, 3)   (n_samples, n_features)
true labels : [0, 1, 2, 3, 4, 5, 6]
n_clusters  : 7


In [3]:
def zscore(x):
    return (x - x.mean(axis=0)) / (x.std(axis=0) + 1e-9)

data_z = zscore(data)
print(f"mean after z-score : {data_z.mean(axis=0).round(4)}")
print(f"std  after z-score : {data_z.std(axis=0).round(4)}")

mean after z-score : [ 0. -0. -0.]
std  after z-score : [1. 1. 1.]


## 2  Train the ESOM

Grid rule of thumb: `5 × sqrt(n_samples)` total neurons is the *minimum* for
the ESOM regime.  For Hepta (212 points) that is ~73 neurons.  We use 32 × 36
= 1152 — about 5.4× the sample count — so cluster topology can emerge clearly.

**Backends** — pick one; the others require an optional install:

| `backend=` | install | training style |
|---|---|---|
| `"minisom"` *(default)* | core dep | sequential, one sample at a time |
| `"sompy"` | `pip install '.[sompy]'` | batch, full dataset per step |
| `"torchsom"` | `pip install '.[torchsom]'` | mini-batch, GPU-capable |

**`iterations`** always counts *total sample presentations* across all backends.
Batch backends convert automatically: sompy uses it as a training-length scale factor;
torchsom converts it to `epochs = max(1, iterations // n_samples)` so that total
data exposure stays comparable to MiniSom.  To pin epochs directly for torchsom,
pass `torchsom_kwargs={"epochs": 50}` — that takes priority over `iterations`.

In [5]:
GRID_X, GRID_Y = 32, 36
N_FEATURES = data_z.shape[1]

#som = ESOM(GRID_X, GRID_Y, N_FEATURES, random_seed=42)                                        # minisom (default)
# som = ESOM(GRID_X, GRID_Y, N_FEATURES, random_seed=42, backend="sompy")                    # pip install '.[sompy]'
som = ESOM(GRID_X, GRID_Y, N_FEATURES, random_seed=42, backend="torchsom")                 # pip install '.[torchsom]'
# som = ESOM(GRID_X, GRID_Y, N_FEATURES, random_seed=42, backend="torchsom",
#            torchsom_kwargs={"epochs": 50})                                                   # pin epochs directly
som.fit(data_z, iterations=15_000)  # ~71 epochs for torchsom (15_000 // 212)

print(f"Weights shape : {som.weights.shape}  (rows, cols, features)")

Weights shape : (32, 36, 3)  (rows, cols, features)


## 3  U-matrix

The U-matrix encodes the distance between each neuron and its neighbours.
Low values (valleys) = cluster cores.  High values (ridges) = cluster boundaries.

In [6]:
u = som.u_matrix()
print(f"U-matrix shape : {u.shape}")
print(f"U-matrix range : [{u.min():.4f}, {u.max():.4f}]")

plot_grid_heatmap(u, "U-matrix — Hepta ESOM (32×36)",
             scheme="greys", value_name="U-distance", cell_pixels=8.0)

U-matrix shape : (32, 36)
U-matrix range : [0.0745, 1.0000]


alt.Chart(...)

## 4  Hit map

The hit map counts how many training samples map to each neuron (Best Matching Unit).
Neurons that attract many samples are cluster cores; empty neurons are either
boundaries or unexplored regions.

In [7]:
hits = som.hit_map(data_z)
print(f"Hit map shape     : {hits.shape}")
print(f"Total hits        : {int(hits.sum())}  (should equal n_samples={len(data_z)})")
print(f"Empty neurons     : {int((hits == 0).sum())} / {hits.size}")
print(f"Max hits per node : {int(hits.max())}")

u_chart = plot_grid_heatmap(u, "U-matrix", scheme="greys", value_name="U-distance")
p_chart = plot_grid_heatmap(hits, "Hit map (count)", scheme="blues", value_name="hits")

alt.hconcat(u_chart, p_chart).resolve_scale(color="independent")

Hit map shape     : (32, 36)
Total hits        : 212  (should equal n_samples=212)
Empty neurons     : 1072 / 1152
Max hits per node : 7


alt.HConcatChart(...)

## 5  BMU indices

`bmu_indices(data)` returns the (row, col) grid position of the Best Matching
Unit for every sample.  Overlaying BMUs on the U-matrix — coloured by true
class — shows how the 7 Hepta clusters map onto the SOM topology.

In [8]:
bmus = som.bmu_indices(data_z)
print(f"BMU indices shape : {bmus.shape}  (n_samples, 2)")
print(f"Sample 0  → SOM node ({bmus[0, 0]}, {bmus[0, 1]})")

bmu_df = pd.DataFrame({
    "col": bmus[:, 1],
    "row": bmus[:, 0],
    "class": true_labels.astype(str),
})

base = plot_grid_heatmap(u, "BMUs coloured by true Hepta label",
                    scheme="greys", value_name="U-distance", cell_pixels=8.0)

scatter = (
    alt.Chart(bmu_df).mark_circle(size=30, opacity=0.75, stroke="white", strokeWidth=0.4)
    .encode(
        x=alt.X("col:O", sort=None),
        y=alt.Y("row:O", sort="descending"),
        color=alt.Color("class:N",
                        scale=alt.Scale(scheme="tableau10"),
                        legend=alt.Legend(title="True class")),
        tooltip=["col", "row", "class"],
    )
)

alt.layer(base, scatter)

BMU indices shape : (212, 2)  (n_samples, 2)
Sample 0  → SOM node (16, 16)


alt.LayerChart(...)

## 6  Component planes

A component plane shows the mean value of one input feature across the SOM grid.
Comparing planes reveals which features drive which regions — essential for
interpreting what each cluster represents.

In [9]:
plot_component_planes(som, data_z, feature_names=["x1", "x2", "x3"], cell_pixels=8.0)

alt.Chart(...)

## Summary

| Object | Shape | What it encodes |
|---|---|---|
| `som.weights` | `(32, 36, 3)` | Prototype vectors for each neuron |
| `som.u_matrix()` | `(32, 36)` | Inter-neuron distances — reveals cluster topology |
| `som.hit_map(data)` | `(32, 36)` | Sample count per neuron — density map |
| `som.bmu_indices(data)` | `(212, 2)` | Grid position of the best-matching unit per sample |
| `som.component_plane(data, i)` | `(32, 36)` | Mean value of feature *i* per neuron |

Next: **02_ustar_clustering.ipynb** — combine U-matrix and hit map into the
U\*-matrix and run U\*F flood clustering to extract cluster labels automatically.